In [ ]:
#!git clone https://github.com/whyhardt/SPICE.git

In [ ]:
# !pip install -e SPICE

In [1]:
import sys

import torch

from spice import SpiceEstimator

sys.path.append("../../..")
from weinhardt2026.utils.benchmarking_gru import GRUModel, training
from weinhardt2026.studies.castro2025.benchmarking_castro2025 import Castro2025Model, get_dataset, generate_behavior
import spice_castro2025, weinhardt2026.studies.castro2025.spice_castro2025 as spice_castro2025

# NOTEBOOK CONFIG

In [21]:
train_spice = False
train_cfs = True
train_gru = False

## Load dataset

In [3]:
path_data = 'data/eckstein2024.csv'
test_sessions = (2,)  # pick sessions that exist for all participants; adjust if needed
dataset_train, dataset_test, info_dataset = get_dataset(path_data=path_data, test_sessions=test_sessions, verbose=True)

Shape of dataset: torch.Size([4158, 150, 1, 13])
Number of participants: 862
Number of actions in dataset: 4


In [4]:
# from spice import SpiceDataset

# # keep only 100 timesteps
# dataset_train = SpiceDataset(dataset_train.xs[:, :100], dataset_train.ys[:, :100])

# # keep only 100 participants for rapid prototyping
# keep_participants = torch.arange(0, 50)

# def keep_subset(dataset, subset):
#     participant_ids = dataset.xs[:, 0, 0, -1]
#     mask = torch.isin(participant_ids, subset)
#     return SpiceDataset(dataset.xs[mask], dataset.ys[mask])

# dataset_train = keep_subset(dataset_train, keep_participants)
# dataset_test = keep_subset(dataset_test, keep_participants)    


## SPICE Setup

## SPICE Training

Let's setup now the `SpiceEstimator` object and fit it to the data! 

We are going to do this in two steps:

1. Without fitting the SINDy coefficients to get the pure RNN performance given the selected architecture. 
2. With fitting SINDy coefficients to get the final performance of the interpretable model

That way we can disentangle the gap between GRU and SPICE w.r.t. architecture and SINDy library 

In [5]:
path_spice = 'params/spice_castro2025_62_stage1.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=spice_castro2025.SpiceModel,
        spice_config=spice_castro2025.CONFIG,
        n_actions=dataset_train.n_actions,
        n_participants=dataset_train.n_participants,
        
        epochs=1000,
        warmup_steps=250,

        verbose=True,
        # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        save_path_spice=path_spice,
    )

In [6]:
if train_spice:
    estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

In [7]:
estimator.load_spice(path_spice)
# estimator.aggregate_coefficients()

In [8]:
# Print example SPICE model for first participant
print("\nExample SPICE model (participant 0):")
estimator.print_spice_model(participant_id=0)


Example SPICE model (participant 0):
value_reward_env[t+1]             = -0.009 1 + 0.964 value_reward_env[t] + 0.023 reward[t] 
value_reward_chosen[t+1]          = -0.432 1 + 0.922 value_reward_chosen[t] + -0.052 reward_env + 0.621 reward[t] + -0.095 value_reward_chosen*reward[t] + 0.012 reward_env^2 + 0.029 reward_env*reward[t] + 0.879 reward[t]^2 
value_reward_not_chosen[t+1]      = -0.044 1 + 0.808 value_reward_not_chosen[t] + 0.034 reward_env + 0.009 value_reward_mean 
value_choice_chosen[t+1]          = -0.356 1 + 0.432 value_choice_chosen[t] + 0.244 action[t-1] + 0.022 value_choice_chosen^2 + 0.054 value_choice_chosen*action[t-1] + 0.244 action[t-1]^2 
value_choice_not_chosen[t+1]      = 0.053 1 + 0.825 value_choice_not_chosen[t] + -0.585 action[t-1] + -0.067 value_choice_not_chosen^2 + -0.745 value_choice_not_chosen*action[t-1] + -0.586 action[t-1]^2 
value_exploration_chosen[t+1]     = -0.786 1 + 0.448 value_exploration_chosen[t] + 2.363 dvalue_pos + -2.634 dvalue_neg + -0.11

## Benchmarking

### Castro2025 benchmark model

In [22]:
path_spice = 'params/spice_castro2025.pkl'

# Benchmark model: Castro et al. 2025
cfs = Castro2025Model(
    n_participants=dataset_train.n_participants,
    n_actions=dataset_train.n_actions,
    batch_first=True,
    )

path_cfs = path_spice.replace('spice_', 'cfs_')

In [23]:
if train_cfs:
    optimizer_cfs = torch.optim.Adam(params=cfs.parameters(), lr=0.01)
    epochs = 1000

    cfs = training(
        model=cfs,
        optimizer=optimizer_cfs,
        dataset_train=dataset_train,
        dataset_test=dataset_test,
        epochs=epochs,
        loss_kwargs={'label_smoothing': 0.0},
    )

    torch.save(cfs.state_dict(), path_cfs)

Epoch 1/1000: L(Train): 0.6391032338142395; L(Test): 0.6491584181785583
Epoch 2/1000: L(Train): 0.6380892992019653; L(Test): 0.648391842842102
Epoch 3/1000: L(Train): 0.6371073722839355; L(Test): 0.6476544141769409
Epoch 4/1000: L(Train): 0.6361557841300964; L(Test): 0.6469435095787048
Epoch 5/1000: L(Train): 0.6352332830429077; L(Test): 0.6462575793266296
Epoch 6/1000: L(Train): 0.6343384981155396; L(Test): 0.645593523979187
Epoch 7/1000: L(Train): 0.6334707736968994; L(Test): 0.6449497938156128
Epoch 8/1000: L(Train): 0.6326279640197754; L(Test): 0.6443249583244324
Epoch 9/1000: L(Train): 0.6318095922470093; L(Test): 0.6437169313430786
Epoch 10/1000: L(Train): 0.6310133934020996; L(Test): 0.6431248188018799
Epoch 11/1000: L(Train): 0.6302382349967957; L(Test): 0.6425477862358093
Epoch 12/1000: L(Train): 0.6294822096824646; L(Test): 0.6419848799705505
Epoch 13/1000: L(Train): 0.6287450194358826; L(Test): 0.6414342522621155
Epoch 14/1000: L(Train): 0.6280249953269958; L(Test): 0.640895

In [11]:
cfs.load_state_dict(torch.load(path_cfs, map_location='cpu'))

<All keys matched successfully>

### GRU Model

In [12]:
gru = GRUModel(
    n_actions=dataset_train.n_actions, 
    additional_inputs=2, 
    dropout=0.1,
    hidden_size=32,
    )
path_gru = path_spice.replace('spice_', 'gru_')

In [13]:
if train_gru:
    epochs = 1000
    optimizer_gru = torch.optim.Adam(gru.parameters(), lr=0.01)

    gru = training(
        model=gru,
        optimizer=optimizer_gru,
        dataset_train=dataset_train,
        dataset_test=dataset_test,
        epochs=epochs,
        loss_kwargs={'label_smoothing': 0.0},
        ).to(torch.device('cpu'))

    torch.save(gru.state_dict(), path_gru)

In [14]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))

<All keys matched successfully>

# ANALYSIS

In [15]:
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals
from analysis_generative import analysis_generative_behavior

In [16]:
estimator.eval()
cfs.eval()
gru.eval()

GRUModel(
  (participant_embedding): Embedding(1, 16)
  (experiment_embedding): Embedding(1, 16)
  (linear_in): Linear(in_features=10, out_features=32, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (gru): GRU(32, 32, batch_first=True)
  (linear_out): Linear(in_features=32, out_features=4, bias=True)
)

## General Analysis

In [17]:
# Dataset-specific behavioral analysis placeholder.
# Replace with Eckstein2024-specific columns if needed.

In [18]:
print("Fitted Castro2025 parameters:")
print("\nBeta_r")
print(cfs.beta_r)
print("\nLapse")
print(cfs.lapse)
print("\nPrior")
print(cfs.prior)
print("\nAlpha exploration rate")
print(cfs.alpha_exploration_rate)
print("\nDecay rate")
print(cfs.decay_rate)
print("\nGamma")
print(cfs.gamma)
print("\nTemperature")
print(cfs.temperature)

Fitted Castro2025 parameters:

Beta_r
tensor([2.6436, 2.5725, 1.6350, 2.6326, 1.4967, 2.2864, 2.0761, 2.0822, 1.3668,
        2.0461, 2.6666, 2.1200, 2.3732, 2.6012, 2.1834, 2.1364, 2.7813, 2.5134,
        2.6401, 3.0772, 2.2167, 2.3841, 1.6499, 5.0067, 1.8361, 1.8675, 1.6592,
        2.4276, 1.9511, 2.3570, 1.9517, 2.0336, 2.0844, 2.6510, 2.3429, 2.4858,
        2.2422, 2.5832, 2.2496, 1.5830, 1.9944, 1.7697, 2.4686, 1.7852, 2.4369,
        2.5515, 2.3137, 1.7513, 2.4290, 2.6330, 2.1942, 1.9395, 2.2016, 1.2631,
        1.6957, 1.7863, 1.7522, 2.4744, 2.4172, 2.4560, 1.7453, 2.0340, 2.3964,
        1.8907, 2.5230, 1.7380, 2.3518, 2.0107, 1.4852, 1.6658, 1.8698, 1.9066,
        3.0647, 2.3329, 1.6459, 2.7516, 2.0881, 1.4873, 2.4971, 3.3913, 2.8240,
        2.1707, 2.4945, 1.8685, 1.6642, 2.5955, 1.7528, 2.7511, 1.6749, 2.8953,
        1.5376, 2.2284, 2.2507, 4.8863, 1.9715, 1.0420, 2.3308, 1.7879, 2.2634,
        2.2010, 2.0965, 1.8153, 2.6210, 2.3455, 2.2507, 2.5202, 1.9719, 2.3137,
  

## Analysis Model Evaluation

In [24]:
analysis_model_evaluation(
    dataset=dataset_train,
    spice_model=estimator,
    benchmark_model=cfs.to(torch.device('cpu')),
    gru_model=gru.eval().to(torch.device('cpu')),
    )

Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


,Trial Lik.,(std),n_parameters,(std),NLL,AIC,BIC
Benchmark,0.621020,0.159128,13.000000,0.000000,232753.859375,465533.71875,465678.0000
GRU,0.627344,0.156140,6852.000000,0.000000,227803.765625,469311.53125,545363.6250
SPICE-RNN,0.602785,0.156805,15948.000000,0.000000,247314.750000,526525.50000,703536.3750
SPICE,0.595553,0.157122,38.266823,2.774654,253211.718750,506500.40625,506927.5625


In [25]:
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    benchmark_model=cfs.to(torch.device('cpu')),
    gru_model=gru.eval().to(torch.device('cpu')),
    )

Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


,Trial Lik.,(std),n_parameters,(std),NLL,AIC,BIC
Benchmark,0.598384,0.161196,13.000000,0.000000,65651.343750,131328.687500,131455.546875
GRU,0.618730,0.157360,6852.000000,0.000000,61376.554688,136457.109375,203322.843750
SPICE-RNN,0.593454,0.156552,15948.000000,0.000000,66708.921875,165313.843750,320943.562500
SPICE,0.584291,0.156896,38.266823,3.530624,68698.375000,137473.281250,137846.718750


## Analysis generative behavior

In [ ]:
estimator.use_sindy(True)
generate_behavior(
    model=estimator,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_spice.csv'
)

estimator.use_sindy(False)
generate_behavior(
    model=estimator,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_spice_rnn.csv'
)

gru.eval()
generate_behavior(
    model=gru,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_gru.csv'
)

generate_behavior(
    model=cfs,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_cfs.csv'
)

In [ ]:
analysis_generative_behavior(
    path_data_real=path_data,
    path_data_gru='data/eckstein2024_gru.csv',
    path_data_benchmark='data/eckstein2024_cfs.csv',
    path_data_spice='data/eckstein2024_spice.csv',
    path_data_spice_rnn='data/eckstein2024_spice_rnn.csv',
    output_dir='results',
)

## Analysis coefficient distributions

In [ ]:
# analysis_coefficients_distributions(
#     spice_model=estimator,
#     output_dir='results',
# )

## Analysis Individual Differences

In [ ]:
# analysis_coefficients_individuals(
#     criterion="SomeCriterionColumnInYourDataset",
#     analysis="disc",  # also: "cont"
#     reference="ReferenceGroupFromCriterionColumn",  # only necessary if analysis="disc"
    
#     path_data=path_file,
    
#     spice_model=estimator,
    
#     dir_output='results',
# )